# Ascon-AEAD128 host demo (no board)
Y03/Y04 offline subset. Run from the repository root or src/pynq. All three modes use the same locked Python reference, NOT hardware execution. Cycles are historical **simulated** warm RV32 mcycle samples, not Python timing or PS/PL end-to-end measurements.
Public KAT keys/nonces below are test-only; never reuse a nonce with the same key in deployment.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/pynq/ascon_host_api.py').is_file())
sys.path.insert(0, str(root / 'src/pynq'))
from ascon_host_api import AsconHost, SimulatedBackend, AuthenticationError, cycle_table
from golden_checker import parse_kat, check_all
host = AsconHost(SimulatedBackend('ise'))
cases = parse_kat()


In [ ]:
print('PT bytes / AD bytes / encrypt / decrypt / cycles / evidence')
for m, a in ((0, 0), (1, 0), (16, 0), (16, 16), (17, 0), (32, 0)):
    c = next(c for c in cases if (len(c['PT']), len(c['AD'])) == (m, a))
    enc = host.encrypt(c['Key'], c['Nonce'], c['AD'], c['PT'])
    dec = host.decrypt(c['Key'], c['Nonce'], c['AD'], c['CT'][:-16], c['CT'][-16:])
    assert enc.ciphertext + enc.tag == c['CT'] and dec.plaintext == c['PT']
    print(m, a, 'PASS', 'PASS', enc.cycles, enc.cycle_info.status)
summary = check_all(host)
assert summary['passed'] == 1089 and summary['failed'] == 0


In [ ]:
c = next(c for c in cases if len(c['PT']) == 16 and not c['AD'])
bad_tag = bytes([c['CT'][-16] ^ 1]) + c['CT'][-15:]
try:
    host.decrypt(c['Key'], c['Nonce'], c['AD'], c['CT'][:-16], bad_tag)
except AuthenticationError:
    print('Tampered tag rejected: PASS; no plaintext returned')
else:
    raise AssertionError('invalid tag accepted')


In [ ]:
table = cycle_table()
print('Historical simulated warm 16B encrypt, AD=0; not board measurements')
for mode, info in table.items():
    print(f'{mode}: {info.value} cycles; sw/{mode} = {table["sw"].value / info.value:.3f}x')
    print('  Scope:', info.scope, '\n  Source:', info.source)
print('MMIO fully offloads AEAD; ISE only offloads permutations. Neither includes PS/Python overhead.')


In [ ]:
# TODO(board): Freeze Y01 protocol/version/registers/buffer ownership and capacity.
# TODO(board): Implement Backend on the verified PYNQ image with bounded polling,
# cache maintenance, request matching, authentication gating and timeout recovery.
# TODO(board): Load matching bit/hwh, then repeat KAT and collect real board cycles.
